<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB09_Clustering_and_Dimensionality_Reduction_with_Real_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NB09 · Class 9 — Clustering and Dimensionality Reduction with Real Data**

## Block 2: AI — Machine Learning (continued)

Every model in `NB07`/`NB08` was **supervised**: we always had a known target (fuel type, fuel consumption, mine vs. rock) to predict. This class switches to **unsupervised learning**, where there is no target at all — only the goal of finding structure in the data on its own. We cover two of the most widely used unsupervised techniques:

- **K-Means clustering**, applied to the real [`ship_fuel_efficiency.csv`](https://github.com/JuanZapa7a/AINavalEngineering/blob/main/Datasets/ship_fuel_efficiency.csv) dataset from `NB07`, to discover voyage "operational profiles" without using any of its labels.
- **Principal Component Analysis (PCA)**, applied to the real [`sonar.all-data`](https://github.com/JuanZapa7a/AINavalEngineering/blob/main/Datasets/sonar.all-data) dataset from `NB08` (60 features!), to visualize high-dimensional data in 2D and to see whether clustering alone can recover the real mine/rock split.

### Learning objectives

By the end of this class, students will be able to:
- Explain what makes a learning problem unsupervised, and when you'd reach for it instead of a supervised model.
- Explain how K-Means assigns points to clusters and updates centroids, and its main limitation (you must choose *k*).
- Use the elbow method and the silhouette score to choose a reasonable number of clusters.
- Profile and interpret clusters by comparing them against known (but unused) labels.
- Explain what PCA does (variance-maximizing projections) and read an explained-variance plot.
- Use PCA to visualize high-dimensional data in 2D, and evaluate clustering quality against ground truth with the Adjusted Rand Index.

### Agenda (2-hour class)

| # | Class segment | Approx. duration | Type |
|---|---------------------|:---:|:---:|
| 1 | Recap of NB01–NB08, today's roadmap | 5 min | Theory |
| 2 | Unsupervised learning: what and why | 10 min | Theory |
| 3 | K-Means: how it works | 15 min | Theory |
| 4 | Hands-on: choosing *k* for real voyage data (elbow method, silhouette score) | 15 min | Practice |
| 5 | Interpreting clusters: profiling and comparing against known labels | 20 min | Practice |
| 6 | Why reduce dimensionality? | 10 min | Theory |
| 7 | PCA: how it works | 15 min | Theory |
| 8 | Hands-on: PCA + clustering on real 60-feature sonar data | 20 min | Practice |
| 9 | Unsupervised learning in naval/ocean engineering (overview) | 5 min | Theory |
| 10 | Summary, homework, next class | 5 min | Theory |

> Timings are approximate guidance, not a strict script — there are no scheduled breaks. If we cover everything with time to spare, class ends early; that can happen and is fine.


---

## 1. Recap: where we are

- **`NB01`–`NB02`**: AI history, Python/Colab/NumPy/Pandas essentials.
- **`NB07`**: the supervised ML workflow — train/val/test, metrics, a classifier and a regressor, cross-validation, data leakage.
- **`NB08`**: classification algorithms in depth — decision trees, ensembles, SVM, leakage-safe pipelines, hyperparameter search.
- **`NB09`** (today): unsupervised learning — finding structure in data with **no target label at all**.

Still inside **Block 2 — AI: Machine Learning** of the course roadmap.

---

## 2. Unsupervised learning: what and why

Every model so far has been trained on **labeled** examples: we always knew the "right answer" (fuel type, fuel consumption, mine vs. rock) for every training row. In many real situations, **there is no label at all** — `either because no one has annotated the data, or because we don't yet know what categories exist`. Unsupervised learning looks for structure in the data itself:

| Task | Goal | Naval/ocean example |
|---|---|---|
| **Clustering** | Group similar records together | Segment a fleet into operational profiles without predefined categories |
| **Dimensionality reduction** | Compress many features into fewer, while keeping most of the information | Reduce 60 sonar frequency bands to 2 for visualization |
| **Anomaly detection** | Flag records that don't fit any normal pattern | Detect an unusual engine reading before it becomes a failure |

> **Further reading**: [Unsupervised learning (Wikipedia)](https://en.wikipedia.org/wiki/Unsupervised_learning).

Today we cover the first two — clustering (K-Means) and dimensionality reduction (PCA) — both applied to real datasets you've already seen.

---

## 3. K-Means: how it works

**K-Means** partitions data into *k* clusters, where *k* is chosen in advance. It works iteratively:

1. Place *k* initial cluster centers ("centroids"), typically at random points.
2. **Assign** every data point to its nearest centroid (by Euclidean distance).
3. **Update** each centroid to the mean of the points now assigned to it.
4. Repeat steps 2–3 until assignments stop changing (convergence).

Two things to keep in mind before using it:
- **You must choose *k* in advance** — unlike supervised learning, there's no "correct" number of clusters written in the data; we have to estimate a reasonable one (Part 4).
- **Distance-based, so scale matters** — exactly like SVM in `NB08`, features on different scales will dominate the distance calculation unless standardized first.

> **Further reading**: [k-means clustering (Wikipedia)](https://en.wikipedia.org/wiki/K-means_clustering) · [`sklearn.cluster.KMeans` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html).

---

## 4. Hands-on: choosing *k* for real voyage data

We return to the real `ship_fuel_efficiency.csv` dataset from `NB07` — but this time, we deliberately **ignore its labels** (`ship_type`, `fuel_type`, `weather_conditions`) and cluster voyages using only three numeric operational features: `distance`, `fuel_consumption`, and `engine_efficiency`. (`CO2_emissions` is left out for the same reason as in `NB07` — it's almost a duplicate of `fuel_consumption`, so including it would just double-count the same information instead of adding a new dimension.)

In [ ]:
!wget -q -O ship_fuel_efficiency.csv https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/ship_fuel_efficiency.csv

import pandas as pd

fuel = pd.read_csv("ship_fuel_efficiency.csv")
fuel.head()

Scale the three clustering features — required for K-Means, exactly as it was for SVM:

In [ ]:
from sklearn.preprocessing import StandardScaler

cluster_features = ["distance", "fuel_consumption", "engine_efficiency"]
X_cluster = fuel[cluster_features]

scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

Now the key question: **how many clusters?** Two common tools:

- The **elbow method**: plot K-Means' `inertia_` (the total squared distance from each point to its assigned centroid) against *k*. Inertia always decreases as *k* grows — we're looking for the point where adding another cluster stops helping much, forming an "elbow" in the curve.
- The **silhouette score**: measures how well-separated the clusters are (from -1 to 1; higher is better). Unlike inertia, it doesn't automatically favor more clusters, so it's a useful second opinion.

> **Further reading**: [elbow method (Wikipedia)](https://en.wikipedia.org/wiki/Elbow_method_%28clustering%29) · [silhouette (Wikipedia)](https://en.wikipedia.org/wiki/Silhouette_%28clustering%29) · [`sklearn.metrics.silhouette_score` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

inertias = []
sil_scores = []
k_range = range(1, 9)

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_cluster_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_cluster_scaled, labels) if k > 1 else None)

Plot both side by side:

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(list(k_range), inertias, marker="o")
axes[0].set_xlabel("k (number of clusters)")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow method")

axes[1].plot(list(k_range)[1:], sil_scores[1:], marker="o", color="darkorange")
axes[1].set_xlabel("k (number of clusters)")
axes[1].set_ylabel("Silhouette score")
axes[1].set_title("Silhouette score by k")

plt.tight_layout()
plt.show()

**Read your own plots**: where does the elbow curve visibly bend? Does the silhouette score peak at the same *k*, or a different one? The two methods don't always agree — when they don't, domain knowledge (how many genuinely different operational profiles would make sense for a real fleet?) should help you pick. For the rest of this section we'll use **k = 3**, but change it and re-run everything below if your plots suggest otherwise.

---

## 5. Interpreting clusters

Fit the final model and attach the cluster label back onto the DataFrame:

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
fuel["cluster"] = kmeans.fit_predict(X_cluster_scaled)

fuel["cluster"].value_counts().sort_index()

A cluster is only useful once we know what it *means*. Profile each cluster by its average feature values — `this is where domain knowledge turns "cluster 0, 1, 2" into an actual operational story`:

In [ ]:
fuel.groupby("cluster")[cluster_features].mean().round(1)

Seeing the clusters directly on two of the three real features makes the profile table concrete:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(fuel["distance"], fuel["fuel_consumption"], c=fuel["cluster"], cmap="viridis", alpha=0.6, s=15)
ax.set_xlabel("Distance (nm)")
ax.set_ylabel("Fuel consumption (L)")
ax.set_title("Real voyages, colored by K-Means cluster (k=3)")
legend1 = ax.legend(*scatter.legend_elements(), title="Cluster")
ax.add_artist(legend1)
plt.show()


**Try it yourself**: the average silhouette score (used to choose *k* in Part 4) hides which *individual* points are borderline. Compute `sklearn.metrics.silhouette_samples` for this `k=3` clustering and find the 5 real voyages with the lowest per-point silhouette — points closest to being equally well-explained by a neighboring cluster.

In [ ]:
from sklearn.metrics import silhouette_samples

fuel["silhouette"] = silhouette_samples(X_cluster_scaled, fuel["cluster"])
borderline = fuel.sort_values("silhouette").head(5)
borderline[["cluster", "silhouette"] + cluster_features]


Now the real test: we clustered **without using `ship_type` at all**. Does the unsupervised grouping line up with the known ship types, or did it discover something different (e.g., grouped by voyage length/intensity instead of vessel category)? A cross-tabulation answers this directly:

In [ ]:
pd.crosstab(fuel["cluster"], fuel["ship_type"])

If each cluster is dominated by one or two ship types, K-Means largely rediscovered a category we already had — a reassuring sanity check. If clusters mix ship types freely, it found a *different*, equally valid pattern (e.g., "short efficient voyages" vs. "long inefficient voyages") that cuts across vessel type. Neither outcome is wrong — unsupervised learning finds whatever structure is strongest in the features you gave it, which may or may not match the category you had in mind.

---

## 6. Why reduce dimensionality?

Our clustering example used 3 features — easy to reason about and even plot directly. The Sonar dataset from `NB08` has **60**. Problems with high-dimensional data:

- **You can't visualize it directly** — humans see in 2 or 3 dimensions, not 60.
- **The curse of dimensionality**: as dimensions grow, data points become sparse and distance-based methods (K-Means, KNN, SVM) become less reliable — most points end up roughly equidistant from each other.
- **Redundancy and noise**: nearby frequency bands in a sensor sweep are often correlated (like `fuel_consumption`/`CO2_emissions` in `NB07`); much of the "information" in 60 columns may really only take a handful of independent dimensions to capture.

> **Further reading**: [curse of dimensionality (Wikipedia)](https://en.wikipedia.org/wiki/Curse_of_dimensionality).

**Dimensionality reduction** addresses all three: `compress many correlated features into fewer, uncorrelated ones that preserve as much of the original information` (variance) as possible.

---

## 7. PCA: how it works

**Principal Component Analysis (PCA)** finds new axes ("principal components") that are:
1. **Linear combinations** of the original features,
2. **Uncorrelated** with each other, and
3. Ordered so that the **first component captures the most variance** (spread) in the data, the second captures the most *remaining* variance, and so on.

Projecting data onto just the first 2–3 components usually keeps most of the meaningful structure, even when the original data has dozens of features — because `real-world features are rarely all independent; PCA finds and exploits that redundancy automatically`.

Each component reports an **explained variance ratio**: the fraction of the total variance it accounts for. Plotting the cumulative sum tells you how many components you'd need to keep to preserve, say, 90% of the original information.

> **Further reading**: [Principal component analysis (Wikipedia)](https://en.wikipedia.org/wiki/Principal_component_analysis) · [`sklearn.decomposition.PCA` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html).

---

## 8. Hands-on: PCA and clustering on real sonar data

Reload the real Sonar (Mines vs. Rocks) dataset from `NB08` — 208 sonar returns, 60 frequency-band features:

In [ ]:
!wget -q -O sonar.csv https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/sonar.all-data

sonar = pd.read_csv("sonar.csv", header=None)
sonar.columns = [f"freq_{i}" for i in range(60)] + ["label"]

X_sonar = sonar.drop(columns="label")
y_sonar = sonar["label"]  # kept aside only to check our unsupervised result later

Scale, then reduce to 2 components purely for visualization:

In [ ]:
from sklearn.decomposition import PCA

X_sonar_scaled = StandardScaler().fit_transform(X_sonar)

pca_2d = PCA(n_components=2, random_state=42)
X_sonar_pca2 = pca_2d.fit_transform(X_sonar_scaled)

print("Explained variance ratio (2 components):", pca_2d.explained_variance_ratio_.round(3))
print("Total variance captured:", pca_2d.explained_variance_ratio_.sum().round(3))

**Try it yourself**: `pca_2d.components_[0]` holds PC1's loadings — how much each of the 60 original frequency bands contributes to that first, most-informative axis. Which real frequency bands dominate PC1?

In [ ]:
loadings = pd.Series(pca_2d.components_[0], index=X_sonar.columns).sort_values(key=abs, ascending=False)
print("Top 5 contributors to PC1 (by absolute loading):")
print(loadings.head(5))


Two components will not capture *all* the original variance from 60 features — that's expected. Let's see how many components we'd need for most of it:

In [ ]:
pca_full = PCA(random_state=42).fit(X_sonar_scaled)
cumulative_variance = pca_full.explained_variance_ratio_.cumsum()

plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker=".")
plt.axhline(0.9, color="red", linestyle="--", label="90% variance")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA — how many components do we actually need?")
plt.legend()
plt.show()

Now visualize the 2D projection, colored by the **true** label — remember, PCA itself never sees `label`, it only sees the 60 feature columns:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for lab, name in [("M", "Mine"), ("R", "Rock")]:
    mask = y_sonar == lab
    ax.scatter(X_sonar_pca2[mask, 0], X_sonar_pca2[mask, 1], label=name, alpha=0.6)
ax.set_xlabel("Principal component 1")
ax.set_ylabel("Principal component 2")
ax.set_title("Sonar data in 2D (PCA), colored by true label")
ax.legend()
plt.show()

If mines and rocks form two visually separable regions even in this compressed 2D view, that's a good sign the classifiers in `NB08` had real structure to work with. Now the real unsupervised test: cluster the (scaled, full 60-feature) data with K-Means into 2 groups, **without ever showing it the true labels**, and see how well the clusters line up with reality using the **Adjusted Rand Index** (1.0 = perfect agreement, 0.0 = no better than random):

In [ ]:
from sklearn.metrics import adjusted_rand_score

kmeans_sonar = KMeans(n_clusters=2, n_init=10, random_state=42)
sonar_clusters = kmeans_sonar.fit_predict(X_sonar_scaled)

ari = adjusted_rand_score(y_sonar, sonar_clusters)
print("Adjusted Rand Index vs. true mine/rock label:", round(ari, 3))
pd.crosstab(sonar_clusters, y_sonar)

Seeing the K-Means cluster assignment plotted the same way as the true-label plot above makes the Adjusted Rand Index concrete — two colors that mostly line up would mean a high ARI; two colors that split the data differently means a low one:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X_sonar_pca2[:, 0], X_sonar_pca2[:, 1], c=sonar_clusters, cmap="coolwarm", alpha=0.6)
ax.set_xlabel("Principal component 1")
ax.set_ylabel("Principal component 2")
ax.set_title(f"Same 2D projection, colored by K-Means cluster (ARI = {ari:.3f})")
plt.show()


**Interpret your own score**: a high ARI would mean the strongest pattern in the raw sonar signal happens to align with "mine vs. rock" — `genuinely useful to know, since it would suggest even a simple, label-free method captures most of the signal`. A low ARI means the strongest natural grouping in the data is something *else* (e.g., aspect angle, signal strength) — which is exactly why `NB08`'s *supervised* classifiers, trained specifically on the mine/rock label, were the right tool for that task. Comparing the two isn't a failure of either method — it shows why you choose supervised vs. unsupervised learning based on whether you have (and trust) a label worth optimizing for.

> **Further reading**: [Rand index / Adjusted Rand Index (Wikipedia)](https://en.wikipedia.org/wiki/Rand_index) · [`sklearn.metrics.adjusted_rand_score` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html).

---

## 9. Unsupervised learning in naval and ocean engineering (overview)

| Technique | Naval/ocean example |
|---|---|
| Clustering | Segmenting a fleet into operational profiles; grouping voyages by route/weather behavior; identifying distinct sea-state regimes from buoy data |
| Dimensionality reduction | Compressing many correlated sensor channels (vibration, temperature, pressure) before feeding them to a monitoring model; visualizing high-dimensional survey or sonar data |
| Anomaly detection (built on both) | Flagging a voyage or sensor reading that doesn't belong to any known cluster, or that reconstructs poorly after PCA compression — an early warning signal before a full supervised failure-prediction model is available |

---

## Class summary

- Unsupervised learning finds structure with no target label — clustering groups similar records; dimensionality reduction compresses correlated features.
- K-Means alternates between assigning points to the nearest centroid and recomputing centroids; you must choose *k*, and features must be scaled first.
- The elbow method and silhouette score help choose *k*, but domain knowledge often has the final say.
- A cluster is only useful once profiled and interpreted — comparing clusters against a known-but-unused label is a good sanity check, whether or not they end up matching.
- PCA finds uncorrelated axes ordered by how much variance they explain, letting us visualize 60-dimensional sonar data in 2D and understand how much information a handful of components actually capture.
- Unsupervised and supervised results can disagree (as our sonar Adjusted Rand Index likely showed) — that's informative, not a failure, and it's exactly why both approaches exist.

## For the next class (NB10)

We'll pull the full workflow together — data curation, feature selection, pipelines, hyperparameter tuning, and final evaluation — into one complete, tuned Machine Learning project on a new real dataset, closing out Block 2 before Block 3 (Deep Learning) begins.

## Homework / Practice Ideas

1. Re-run Part 4–5 with a different *k* (try both one value smaller and one value larger than what you chose in class) — how does the cluster profile table in Part 5 change?
2. Repeat the Part 4 clustering, but add `CO2_emissions` back into `cluster_features` despite the leakage-style redundancy warning — does it noticeably change the clusters found? Why or why not, given how correlated it is with `fuel_consumption`?
3. In Part 8, try `n_components=3` instead of 2 for the sonar PCA visualization (you'll need a 3D plot, or three 2D pairwise plots) — does the extra dimension visibly separate mines from rocks any better?
4. Cluster the sonar data into `k=3` or `k=4` instead of 2, and cross-tabulate against the true label as we did — with more clusters than true classes, what patterns do you see?
5. Pick any dataset from an earlier class (`NB02`'s `Naval_Dataset.csv`, `NB07`'s ship fuel data) and apply K-Means to it using different feature combinations than we've used so far — what operational groupings, if any, emerge?

> ***As always: a cluster or a principal component is only useful once you can explain, in plain naval-engineering terms, what it actually represents.***

> For a deeper, textbook-level treatment of everything covered across this Block 2 sequence (regression, classification, trees, SVM, clustering, PCA) in one place, see the free textbook [*An Introduction to Statistical Learning*](https://www.statlearning.com/) (James, Witten, Hastie & Tibshirani).
